**ISP: CEO ATTRACTIVENESS AND EQUITY RESEARCH COVERAGE**

**ONE STOP SHOP**

In [3]:
# --- 1. SETUP AND DEPENDENCIES ---
print("Installing dependencies...")
# We install 'pytorchcv' to get the specific SE-ResNeXt architecture
!pip install Pillow torch torchvision opencv-python dlib pytorchcv --quiet

import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from google.colab import files
import os
import cv2
import dlib
import numpy as np
from collections import OrderedDict

# Import the specific model builder from pytorchcv
from pytorchcv.model_provider import get_model as ptcv_get_model

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Define file path
WEIGHTS_FILE = 'ComboNet_SCUTFBP5500.pth'

# Download Dlib landmark model (needed for face alignment)
LANDMARK_MODEL_URL = 'http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2'
LANDMARK_MODEL_FILE = 'shape_predictor_68_face_landmarks.dat'
if not os.path.exists(LANDMARK_MODEL_FILE):
    print("\nDownloading Dlib landmark model...")
    !wget -nc {LANDMARK_MODEL_URL} -O {LANDMARK_MODEL_FILE}.bz2
    !bzip2 -dkf {LANDMARK_MODEL_FILE}.bz2

# Initialize Dlib components
face_detector = dlib.get_frontal_face_detector()
landmark_predictor = dlib.shape_predictor(LANDMARK_MODEL_FILE)

# --------------------------------------------------------------------------------
# --- 2. THE CORRECT MODEL ARCHITECTURE (ComboNet) ---
# --------------------------------------------------------------------------------

class ComboNet(nn.Module):
    def __init__(self):
        super(ComboNet, self).__init__()

        # 1. The Backbone (SEResNeXt-50 from pytorchcv)
        self.backbone = ptcv_get_model("seresnext50_32x4d", pretrained=False)
        self.backbone.output = nn.Identity() # Get raw 2048 features

        # 2. The Branches (fixed to 2048 input, 1 output for regression, 5 for classification)
        self.regression_branch = nn.Linear(2048, 1)
        self.classification_branch = nn.Linear(2048, 5)

    def forward(self, x):
        features = self.backbone(x)
        score = self.regression_branch(features)
        cls = self.classification_branch(features)
        # Returns both score (regression) and cls (classification logits)
        return score, cls

# --------------------------------------------------------------------------------
# --- 3. ROBUST WEIGHT LOADER (Definitive Key Mapping) ---
# FIX: Only includes parameter keys to avoid "Unexpected key(s)" error.
# --------------------------------------------------------------------------------

def load_combonet_weights(model, weights_path, device):
    """Loads weights by filtering and remapping keys to match the model structure."""
    global load_success
    load_success = False

    if not os.path.exists(weights_path):
        print(f"\n❌ ERROR: Weights file '{weights_path}' not found!")
        return model

    try:
        # Load the raw checkpoint
        checkpoint = torch.load(weights_path, map_location=device, encoding='latin1')

        # Find the state dictionary in various save formats
        state_dict = checkpoint.get('state_dict', checkpoint.get('model', checkpoint))

        # Suffixes that indicate an actual tensor parameter (weight, bias, etc.)
        tensor_suffixes = ('.weight', '.bias', '.running_mean', '.running_var')

        # --- DEFINITIVE KEY REMAPPING ---
        new_state_dict = OrderedDict()
        for k, v in state_dict.items():

            # CRITICAL FILTER: Skip all keys that are not actual parameters (solves the module key error)
            if not k.endswith(tensor_suffixes):
                continue

            name = k

            # 1. Handle 'module.' prefix
            if name.startswith('module.'):
                name = name[7:]

            # 2. Handle head keys: 'backbone.regression_branch' -> 'regression_branch'
            if name.startswith('backbone.regression_branch') or name.startswith('backbone.classification_branch'):
                name = name[9:] # removes 'backbone.' prefix

            # 3. Handle backbone keys: 'backbone.init_block' -> 'backbone.features.init_block'
            # The model uses the '.features.' layer, but the file keys do not.
            elif name.startswith('backbone.'):
                if '.features.' not in name:
                    # Insert 'features.' after 'backbone.'
                    name = name[:9] + 'features.' + name[9:]

            new_state_dict[name] = v

        # Load weights. Strict=True ensures perfect matching after remapping.
        model.load_state_dict(new_state_dict, strict=True)
        print(f"✅ Successfully loaded ComboNet weights from {weights_path}")
        load_success = True

    except Exception as e:
        print(f"❌ Error loading weights: {e}")

    return model

# --------------------------------------------------------------------------------
# --- 4. PREPROCESSING AND SCORING ---
# --------------------------------------------------------------------------------

def align_face(image_path):
    """Detects and aligns face using Dlib."""
    image_bgr = cv2.imread(image_path)
    if image_bgr is None: raise FileNotFoundError(f"Cannot read {image_path}")
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    faces = face_detector(image_rgb, 1)
    if not faces:
        print(f"Warning: No face detected in {image_path}. Using center crop.")
        h, w = image_rgb.shape[:2]
        crop_size = min(h, w) // 2
        c_x, c_y = w // 2, h // 2
        return Image.fromarray(image_rgb).crop((c_x - crop_size, c_y - crop_size, c_x + crop_size, c_y + crop_size))

    face = max(faces, key=lambda rect: rect.area())
    landmarks = landmark_predictor(image_rgb, face)
    aligned_face_np = dlib.get_face_chip(image_rgb, landmarks, size=256)
    return Image.fromarray(aligned_face_np)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def get_score(image_path, model, device):
    try:
        aligned_img = align_face(image_path)
        tensor = transform(aligned_img).unsqueeze(0).to(device)
        model.eval()
        with torch.no_grad():
            # Model returns (score, cls)
            output, _ = model(tensor)

        # The model's regression output is clamped (1-5) and scaled to 1-10.
        score = torch.clamp(output, 1.0, 5.0).item()
        return score
    except Exception as e:
        print(f"Error scoring {image_path}: {e}")
        return None

# --------------------------------------------------------------------------------
# --- 5. EXECUTION ---
# --------------------------------------------------------------------------------

print("\n--- Model Initialization ---")
model = ComboNet().to(device)
# load_success flag is set inside the function
load_combonet_weights(model, WEIGHTS_FILE, device)

# If the weights loaded successfully, proceed to scoring
if globals().get('load_success', False):
    print(f"\nPlease ensure '{WEIGHTS_FILE}' is uploaded.")
    print("Upload your CEO images now...")
    uploaded = files.upload()

    print("\n--- SCORING RESULTS (Scale 1-10) ---")
    for filename in uploaded.keys():
        # Skip the weights file if it was uploaded in this step
        if filename == WEIGHTS_FILE or filename == LANDMARK_MODEL_FILE: continue

        score = get_score(filename, model, device)
        if score:
            print(f"• Image: {filename} -> Score: **{score}**")
else:
     print("\nAborting scoring process due to weight loading failure.")

Installing dependencies...
Using device: cpu

--- Model Initialization ---
✅ Successfully loaded ComboNet weights from ComboNet_SCUTFBP5500.pth

Please ensure 'ComboNet_SCUTFBP5500.pth' is uploaded.
Upload your CEO images now...


Saving passport.jpg to passport.jpg

--- SCORING RESULTS (Scale 1-10) ---
• Image: passport.jpg -> Score: **3.671928882598877**
